This notebook is a template for generating a fast PSF emulator.

# Preliminaries/setup

In [ ]:
import time
import datetime
from pathlib import Path

import numpy as np

from matplotlib import pyplot as plt
from matplotlib import animation

from IPython import display

from astropy import units as u

from jwst import datamodels

from nirspectroperf.psf import NIRSpecPSF, NIRSpecPSFINREmulator

import jax
from jax import numpy as jnp
import jaxon

In [ ]:
random_key_seed = 42  # can be None to use the time

template_data_file_fn = '../ngc6791_cals/jw02609010001_03101_00001_nrs1_cal.fits'
template_slit_number = 49

cache_fn = 'psf_cache.npz'  # None to not use the cache

device_dtype = jnp.float32

psf_oversampling = 4
psf_fov = 50
npsfs = 2048

grid_type = 'random' # 'random' or 'grid'
slit_fraction = 1.1 # how much of the slit to sample

hidden_size = 128
num_hidden_layers = 2
s0 = 10.
w0 = 20.

nepochs = 50000
batch_size = 2048
ncycles = 2
minlr = 1e-4
peaklr = 3e-3
warmup_fraction = 0.1
loss_eps = 0
loss_meantologweight = 0.5 # can be 'progressive'

save_model_fn_pattern = 'nirspec_psf_estimator_{dt}.hdf5'  # use "{dt}" to get a timestamp

n_to_show = 4
plot_save_fn_pattern = None  # use "{subname}" to get the plot-specific part or None to skip saving

pbar = True # the type of progress bar to use

In [ ]:
if random_key_seed is None:
    random_key_seed = time.time_ns()
jrng = jax.random.key(random_key_seed)

cache_path = Path(cache_fn) if cache_fn is not None else None

In [ ]:
if pbar is True:
    from tqdm.auto import tqdm
    pbar = tqdm
elif pbar == 'simple':
    from tqdm import tqdm
    pbar = tqdm
elif pbar == 'notebook':
    from tqdm.notebook import tqdm
    pbar = tqdm
elif not pbar:
    pbar = None

# the else is to just leave it as is and hope the user knows what they are doing

In [ ]:
def save_figure_maybe(fig, subname):
    if plot_save_fn_pattern is None:
        return None
    else:
        fn = plot_save_fn_pattern.format(subname=subname)
        fig.savefig(fn)
        return fn

# Load/generate training data

In [ ]:
dm0 = datamodels.open(template_data_file_fn)
slit = dm0.slits[template_slit_number]

In [ ]:
# Use this to bust the cache and force regeneration of the PSF if needed

# if cache_path.exists():
#     cache_path.unlink()

In [ ]:
if cache_path is None or not cache_path.is_file():
    nrs_psf = NIRSpecPSF(observation_template_dm=dm0, 
                        slit=slit, 
                        oversampling=psf_oversampling, 
                        fov=psf_fov)
    if grid_type == 'random':
        psfs, xpx, ypx, xos, yos = nrs_psf.make_psf_points(npsfs, 1*u.micron, slit_fraction=slit_fraction)
    elif grid_type == 'grid':
        psfs, xpx, ypx, xos, yos = nrs_psf.make_psf_grid(npsfs, 1*u.micron, slit_fraction=slit_fraction)
    else:
        raise ValueError(f'Unknown grid_type: {grid_type}')
    psfs = np.array(psfs, dtype=device_dtype)
    xpx = np.array(xpx, dtype=device_dtype)
    ypx = np.array(ypx, dtype=device_dtype)
    psfxs = np.array(xos, dtype=device_dtype)
    psfys = np.array(yos, dtype=device_dtype)

    if cache_path is not None:
        print('Saving to cache file', cache_path)
        np.savez(cache_path, psflux=psfs, xpx=xpx, ypx=ypx, xoffs=psfxs, yoffs=psfys,
                 oversampling=psf_oversampling, fov=psf_fov, grid_type=grid_type, slit_fraction=slit_fraction,
                 example_header=dict(nrs_psf.last_header))

print('Loading from cache file', cache_path)
cache = np.load(cache_path, allow_pickle=True)

if len(cache['psflux']) != npsfs:
    raise ValueError('loaded/generated PSF is the wrong size!  Probably need to delete an old cache')

if (cache.get('oversampling', psf_oversampling) != psf_oversampling or 
    cache.get('fov', psf_fov) != psf_fov or 
    cache.get('grid_type', grid_type) != grid_type or 
    cache.get('slit_fraction', slit_fraction) != slit_fraction):
    raise ValueError('loaded/generated PSF has different oversampling or FOV or slit fraction than expected!  Probably need to delete an old cache')

psfs = jnp.array(cache['psflux'], dtype=device_dtype)
psfxoffs = jnp.array(cache['xoffs'], dtype=device_dtype)
psfyoffs = jnp.array(cache['yoffs'], dtype=device_dtype)
xpx = jnp.array(cache['xpx'], dtype=device_dtype)
ypx = jnp.array(cache['ypx'], dtype=device_dtype)

Render a few of the psfs just to be sure they make sense

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 12))

idxs = np.random.choice(np.arange(psfs.shape[0]), size=4, replace=False)
for idx, ax in zip(idxs, axs.ravel()):
    ax.imshow(np.log(psfs[idx]), origin='lower', interpolation='nearest')
    ax.set_title(f"PSF {idx}, x={psfxoffs[idx]:.3f}, y={psfyoffs[idx]:.3f}")


# Train Model

In [ ]:
jrng, rkey = jax.random.split(jrng)

training_kwargs = {k:globals()[k] for k in ['nepochs', 'batch_size', 'ncycles', 'minlr', 'peaklr', 'warmup_fraction', 'loss_meantologweight', 'loss_eps', 'pbar']}

st = time.time()
mod, training_info = NIRSpecPSFINREmulator.train(
    psfs, psfxoffs, psfyoffs, 
    **training_kwargs, 
    rkey=rkey, xypx=(xpx, ypx),
    hidden_size=hidden_size, num_hidden_layers=num_hidden_layers, s0=s0, w0=w0)
et = time.time()

plt.semilogy(training_info['losses'])
plt.semilogy(training_info['learning_rates'], color='C2', alpha=.5, ls=':')

print(f'Training time: {et - st} seconds')

# Show outputs

In [ ]:
show_choices = np.random.choice(len(psfs), n_to_show, replace=False)
show_choices

## Individual PSFs

Compare the model outputs to some true PSFs

In [ ]:
imshowkwargs = dict(origin='lower', interpolation='nearest')

for idx in show_choices:
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    truen = true/np.sum(true)
    predn = pred/np.sum(pred)

    truemax = np.where(true == true.max())

    maxmax = max(predn.max(), truen.max())

    fig, axs = plt.subplots(2, 3, figsize=(16,14))
    i = axs[0,0].imshow(predn, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 0], orientation='horizontal')
    axs[0,0].set_title(f'Predicted PSF # {idx} x={psfxoffs[idx]:.3f} y={psfyoffs[idx]:.3f}')
    i = axs[0,1].imshow(truen, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 1], orientation='horizontal')
    axs[0,1].set_title('Normalized True PSF')
    i = axs[0,2].imshow(predn-truen, vmin=-maxmax/2, vmax=maxmax/2, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 2], orientation='horizontal')
    axs[0,2].set_title('Difference (Predicted - True)')

    i = axs[1, 0].imshow(predn, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 0], orientation='horizontal')
    axs[1, 0].set_title('Predicted PSF')
    i = axs[1, 1].imshow(truen, vmin=0, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 1], orientation='horizontal')
    axs[1, 1].set_title('Normalized True PSF')
    i = axs[1, 2].imshow(predn-truen, vmin=-maxmax, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 2], orientation='horizontal')
    axs[1, 2].set_title('Difference (Predicted - True)')

    boxhsize = 5
    for ax in axs[1]:
        ax.set_ylim(truemax[0][0]-boxhsize, truemax[0][0]+boxhsize)
        ax.set_xlim(truemax[1][0]-boxhsize, truemax[1][0]+boxhsize)

    fig.tight_layout()

    save_figure_maybe(fig, f'psf_comparison_{idx}')

Same but for the log

In [ ]:
imshowkwargs = dict(origin='lower', interpolation='nearest')

for idx in show_choices:
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    truen = true/np.sum(true)
    predn = pred/np.sum(pred)
    ltruen = jnp.log(truen)
    lpredn = jnp.log(predn)


    truemax = np.where(true == true.max())
    maxmax = max(lpredn.max(), ltruen.max())
    minmin = min(lpredn.min(), ltruen.min())

    fig, axs = plt.subplots(2, 3, figsize=(16, 14))
    i = axs[0,0].imshow(lpredn, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 0], orientation='horizontal')
    axs[0,0].set_title(f'Predicted Log PSF # {idx} x={psfxoffs[idx]:.3f} y={psfyoffs[idx]:.3f}')
    i = axs[0,1].imshow(ltruen, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 1], orientation='horizontal')
    axs[0,1].set_title('Normalized True LogPSF')
    i = axs[0,2].imshow(lpredn-ltruen, **imshowkwargs)
    plt.colorbar(i, ax=axs[0, 2], orientation='horizontal')
    axs[0,2].set_title('Difference (log Pred - log True)')

    i = axs[1, 0].imshow(lpredn, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 0], orientation='horizontal')
    axs[1, 0].set_title('Predicted Log PSF')
    i = axs[1, 1].imshow(ltruen, vmin=minmin, vmax=maxmax, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 1], orientation='horizontal')
    axs[1, 1].set_title('Normalized True Log PSF')
    i = axs[1, 2].imshow(lpredn-ltruen, **imshowkwargs)
    plt.colorbar(i, ax=axs[1, 2], orientation='horizontal')
    axs[1, 2].set_title('Difference (log Pred - log True)')

    boxhsize = 5
    for ax in axs[1]:
        ax.set_ylim(truemax[0][0]-boxhsize, truemax[0][0]+boxhsize)
        ax.set_xlim(truemax[1][0]-boxhsize, truemax[1][0]+boxhsize)

    fig.tight_layout()

    save_figure_maybe(fig, f'logpsf_comparison_{idx}')

## Animated PSF walk

In [ ]:
intervalms = 200
vmax = 0.035
fig, ax = plt.subplots(figsize=(5, 5))

pxscl = 0.026081967499999997
if 'example_header' in cache:
    pxscl = cache['example_header'].item()['PIXELSCL']
pxwid = NIRSpecPSF.msa_slit_width.to_value(u.arcsec)/pxscl
pxhei = NIRSpecPSF.msa_slit_height.to_value(u.arcsec)/pxscl

ax.add_artist(plt.Rectangle((100-pxwid/2, 100-pxhei/2), pxwid, pxhei, fill=False, alpha=0.5, color='r'))

def fanim(xylog):
    x, y, dolog = xylog
    thispsf = mod.evaluate_psf(x, y, 1)
    logstr = ''
    thisvmax = vmax
    if dolog:
        thispsf = np.log(thispsf)
        logstr = 'logflux, '
        thisvmax = np.log(vmax)
    ax.imshow(thispsf, **imshowkwargs, vmax=thisvmax)
    ax.set_title(f'{logstr}X={x:.3f}, Y={y:.3f}')

    fig.tight_layout()

xs = np.linspace(-NIRSpecPSF.msa_slit_width.to_value(u.arcsec)/2, 
                  NIRSpecPSF.msa_slit_width.to_value(u.arcsec)/2, 
                 6)
ys = np.linspace(-NIRSpecPSF.msa_slit_height.to_value(u.arcsec)/2, 
                  NIRSpecPSF.msa_slit_height.to_value(u.arcsec)/2, 
                 6)[::-1]
xys = []
xyslog = []
for i, y in enumerate(ys):
    for x in xs[::i % 2 * 2 -1]:
        xys.append((x, y, False))
        xyslog.append((x, y, True))

anim = animation.FuncAnimation(fig, fanim, frames=xys, interval=intervalms)
display.display_html(anim.to_html5_video(), raw=True)


animlog = animation.FuncAnimation(fig, fanim, frames=xyslog, interval=intervalms)
display.display_html(animlog.to_html5_video(), raw=True)

plt.close(fig)

## Diff statistics of selected PSFs

In [ ]:
fig, axs = plt.subplots(len(show_choices), 2, figsize=(10, 5*len(show_choices)))

for i, idx in enumerate(show_choices):
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    truen = true/np.sum(true)
    predn = pred/np.sum(pred)

    diff = (predn - truen)*predn.size # mean pixel value of the actual pixels is 1

    axs[i, 0].ecdf(diff.ravel())
    axs[i, 0].set_xlabel('diff (normalized pred - true)')
    axs[i, 0].set_xlim(-1, 1)
    axs[i, 0].set_title(f'idx: {idx}')

    axs[i, 1].ecdf(np.abs(diff).ravel())
    axs[i, 1].set_xlabel('abs diff (normalized pred - true)')
    axs[i, 1].semilogx()
    axs[i, 1].set_title(f'xo={psfxoffs[idx]:.3f} yo={psfyoffs[idx]:.3f}')

    print(f'idx: {idx}')
    print('total diff:', diff.sum(), 'mean diff:', diff.mean(), 'diff std:', diff.std())

    qs = [.001, .01, .1,0.25, 0.5, 0.75, .9, .99, .999]
    for q in qs:
        print(f'{q:>10}', end=' ')
    print('')
    for q in qs:
        print(f'{np.quantile(diff, q):+.3e}', end=' ')
    print('\n')

fig.tight_layout()

save_figure_maybe(fig, f'perpsf_comparison_statistics')

## Summary statistics of all PSF diffs

In [ ]:
sums = []
abssums = []
means = []
stds = []
quantiles = []

for idx in range(len(psfs)):
    pred = mod.evaluate_psf(psfxoffs[idx],psfyoffs[idx], 1)
    true = psfs[idx]
    truen = true/np.sum(true)
    predn = pred/np.sum(pred)

    diff = (predn - truen)*predn.size # mean pixel value of the actual pixels is 1

    sums.append(diff.sum())
    abssums.append(np.abs(diff).sum())
    means.append(diff.mean())
    stds.append(diff.std())
    quantiles.append([np.quantile(diff, q) for q in [.001, .01, .1,0.25, 0.5, 0.75, .9, .99, .999]])


fig, axs = plt.subplots(2, 2, figsize=(12, 12))
for ax, data, nm in zip(axs.ravel(), [sums, abssums, means, stds], ['Sum of diff', 'Sum of absolute diff', 'Mean of diff', 'SD of diff']):
    ax.hist(data, bins='auto', log=True)
    ax.set_ylabel(nm)
    ax.set_title(f'mean={np.mean(data):.3}, std={np.std(data):.3}')

fig.tight_layout()

save_figure_maybe(fig, f'allpsf_comparison_statistics')


fig, axs = plt.subplots(3, 3, figsize=(12, 12))

for ax, data, nm in zip(axs.ravel(), np.array(quantiles).T, ['0.1%', '1%', '10%', '25%', '50%', '75%', '90%', '99%', '99.9%']):
    ax.hist([q for q in data], bins='auto', log=True)
    ax.set_xlabel(nm + ' of diff')
    ax.set_title(f'mean={np.mean([q for q in data]):.3}, std={np.std([q for q in data]):.3}')

fig.tight_layout()

save_figure_maybe(fig, f'allpsf_comparison_quantiles')

predn.size

# Timing

Do a timing test to ensure with these network parameters execution doesn't take too long.

In [ ]:
ntest = 15
inp = (psfxoffs[idx], psfyoffs[idx], 1)
dts = []
for _ in range(ntest):
    st = time.time()
    pred = mod.evaluate_psf(*inp).block_until_ready()
    et = time.time()
    dts.append(et - st)

msmean = np.mean(dts)*1000
msstd = np.std(dts)*1000
print(f'Average PSF evaluation time: {msmean:.3} +/- {msstd:.3} ms')

# Save the result

In [ ]:
if save_model_fn_pattern is None:
    print("Not saving model.")
else:
    save_path = Path(save_model_fn_pattern.format(dt=datetime.datetime.now().isoformat().split('.')[0].replace(':','-')))
    jaxon.save(save_path, mod, allow_dill=True)
    print('Saving model to:', save_path.absolute(), 'size=', save_path.stat().st_size/1024/1024, 'MB')

Load with something like:

```
mod = jaxon.load(save_path, allow_dill=True)
```